In [1]:
from pyspark.sql import SparkSession, functions as F

In [2]:
spark = (
    SparkSession.builder.appName("eda_ports_bronze")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262",
    )
    .config("spark.jars.ivy", "/tmp/.ivy2")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.EnvironmentVariableCredentialsProvider",
    )
    .getOrCreate()
)

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [4]:
df = spark.read.parquet("s3a://g9-ais-project/bronze/ports_activity/ports/")
df.printSchema()

root
 |-- date: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day: string (nullable = true)
 |-- portid: string (nullable = true)
 |-- portname: string (nullable = true)
 |-- country: string (nullable = true)
 |-- ISO3: string (nullable = true)
 |-- portcalls_container: string (nullable = true)
 |-- portcalls_dry_bulk: string (nullable = true)
 |-- portcalls_general_cargo: string (nullable = true)
 |-- portcalls_roro: string (nullable = true)
 |-- portcalls_tanker: string (nullable = true)
 |-- portcalls_cargo: string (nullable = true)
 |-- portcalls: string (nullable = true)
 |-- import_container: string (nullable = true)
 |-- import_dry_bulk: string (nullable = true)
 |-- import_general_cargo: string (nullable = true)
 |-- import_roro: string (nullable = true)
 |-- import_tanker: string (nullable = true)
 |-- import_cargo: string (nullable = true)
 |-- import: string (nullable = true)
 |-- export_container: string (nullable = true)
 |-- export_dry_bulk: string (n

In [5]:
print("rows:", df.count())

rows: 5907965


In [7]:
df.show(5, truncate=False)

+----------+-----+---+--------+--------+-------+----+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+--------------------+-----------+-------------+------------+------+----------------+---------------+--------------------+-----------+-------------+------------+------+--------+----+
|date      |month|day|portid  |portname|country|ISO3|portcalls_container|portcalls_dry_bulk|portcalls_general_cargo|portcalls_roro|portcalls_tanker|portcalls_cargo|portcalls|import_container|import_dry_bulk|import_general_cargo|import_roro|import_tanker|import_cargo|import|export_container|export_dry_bulk|export_general_cargo|export_roro|export_tanker|export_cargo|export|ObjectId|year|
+----------+-----+---+--------+--------+-------+----+-------------------+------------------+-----------------------+--------------+----------------+---------------+---------+----------------+---------------+---------------

In [8]:
num_cols = [c for c in df.columns if c.startswith(("portcalls", "import", "export"))]

# 1. nulls/empties per column, and values that fail a numeric cast
checks = []
for c in df.columns:
    checks.append(F.sum((F.col(f"`{c}`").isNull() | (F.trim(F.col(f"`{c}`").cast("string")) == "")).cast("int")).alias(f"{c}__null"))
for c in num_cols:
    checks.append(F.sum((F.col(f"`{c}`").isNotNull() & F.col(f"`{c}`").cast("double").isNull()).cast("int")).alias(f"{c}__badnum"))
df.agg(*checks).show(vertical=True)

-RECORD 0------------------------------
 date__null                      | 0   
 month__null                     | 0   
 day__null                       | 0   
 portid__null                    | 0   
 portname__null                  | 0   
 country__null                   | 0   
 ISO3__null                      | 0   
 portcalls_container__null       | 0   
 portcalls_dry_bulk__null        | 0   
 portcalls_general_cargo__null   | 0   
 portcalls_roro__null            | 0   
 portcalls_tanker__null          | 0   
 portcalls_cargo__null           | 0   
 portcalls__null                 | 0   
 import_container__null          | 0   
 import_dry_bulk__null           | 0   
 import_general_cargo__null      | 0   
 import_roro__null               | 0   
 import_tanker__null             | 0   
 import_cargo__null              | 0   
 import__null                    | 0   
 export_container__null          | 0   
 export_dry_bulk__null           | 0   
 export_general_cargo__null      | 0   


In [9]:
# 2. rows per year and date range
df.groupBy("year").agg(F.count("*").alias("rows"), F.min("date").alias("min_date"), F.max("date").alias("max_date")).orderBy("year").show(50)

+----+------+----------+----------+
|year|  rows|  min_date|  max_date|
+----+------+----------+----------+
|2019|753725|2019/01/01|2019/12/31|
|2020|755790|2020/01/01|2020/12/31|
|2021|753725|2021/01/01|2021/12/31|
|2022|753725|2022/01/01|2022/12/31|
|2023|753725|2023/01/01|2023/12/31|
|2024|755790|2024/01/01|2024/12/31|
|2025|753725|2025/01/01|2025/12/31|
|2026|627760|2026-07-10|2026/07/24|
+----+------+----------+----------+



In [10]:
# 3. duplicates on the natural key (portid + date)
dups = df.groupBy("portid", "date").count().filter("count > 1")
print("duplicate keys:", dups.count())
dups.orderBy(F.desc("count")).show(10)

duplicate keys: 0
+------+----+-----+
|portid|date|count|
+------+----+-----+
+------+----+-----+



In [11]:
# 4. distinct ports and ObjectId uniqueness
print("distinct ports:", df.select("portid").distinct().count())
print("distinct ObjectId:", df.select("ObjectId").distinct().count())

distinct ports: 2065
distinct ObjectId: 5709569


In [6]:
d = df.withColumn(
    "date_parsed",
    F.coalesce(F.to_date("date", "yyyy-MM-dd"), F.to_date("date", "yyyy/MM/dd"))
)

# 1. unparsed rows, true date range
d.agg(
    F.sum(F.col("date_parsed").isNull().cast("int")).alias("unparsed"),
    F.min("date_parsed").alias("min_date"),
    F.max("date_parsed").alias("max_date"),
).show()

+--------+----------+----------+
|unparsed|  min_date|  max_date|
+--------+----------+----------+
|       0|2019-01-01|2026-10-16|
+--------+----------+----------+



In [13]:
# 2. which format each year uses
d.withColumn("fmt", F.when(F.col("date").contains("/"), "slash").otherwise("dash")) \
 .groupBy("year", "fmt").count().orderBy("year", "fmt").show(30)

+----+-----+------+
|year|  fmt| count|
+----+-----+------+
|2019|slash|753725|
|2020|slash|755790|
|2021|slash|753725|
|2022|slash|753725|
|2023|slash|753725|
|2024|slash|755790|
|2025|slash|753725|
|2026| dash|204435|
|2026|slash|423325|
+----+-----+------+



In [14]:
# 3. duplicates again, on the parsed date
print("dups:", d.groupBy("portid", "date_parsed").count().filter("count > 1").count())

dups: 30975


In [15]:
# 4. does year/month/day agree with the parsed date?
print("mismatch:", d.filter(
    (F.year("date_parsed") != F.col("year")) |
    (F.month("date_parsed") != F.col("month").cast("int")) |
    (F.dayofmonth("date_parsed") != F.col("day").cast("int"))
).count())

mismatch: 0


In [ ]:
d2 = d.withColumn("src", F.when(F.col("date").contains("/"), "csv").otherwise("api"))
num_cols = [c for c in df.columns if c.startswith(("portcalls", "import", "export"))]

a = d2.filter("src = 'api'").select("portid", "date_parsed", F.concat_ws("|", *num_cols).alias("api_vals"))
c = d2.filter("src = 'csv'").select("portid", "date_parsed", F.concat_ws("|", *num_cols).alias("csv_vals"))
j = a.join(c, ["portid", "date_parsed"])

print("overlapping pairs:", j.count())
diff = j.filter("api_vals != csv_vals")
print("pairs with different values:", diff.count())
diff.show(10, truncate=False)

overlapping pairs: 30975


In [25]:
from pyspark.sql import functions as F

s = (spark.read.parquet("s3a://g9-ais-project/silver/ports_activity/ports/")
          .filter("year >= 2025").cache())
s.count()   # loads it into memory once

for p in ["import", "export"]:
    parts = sum(F.col(f"{p}_{k}") for k in ["container", "dry_bulk", "general_cargo", "roro"])
    x = (s.withColumn("d_cargo", F.col(f"{p}_cargo") - parts)
          .withColumn("d_total", F.col(f"{p}_total") - (F.col(f"{p}_cargo") + F.col(f"{p}_tanker"))))
    for dcol in ["d_cargo", "d_total"]:
        print(p, dcol)
        x.filter(F.col(dcol) != 0).agg(
            F.count("*").alias("n"),
            F.sum((F.col(dcol) > 0).cast("int")).alias("positive"),
            F.sum((F.col(dcol) < 0).cast("int")).alias("negative"),
            F.min(dcol).alias("min"),
            F.max(dcol).alias("max"),
            F.expr(f"percentile_approx(abs({dcol}), array(0.5, 0.9, 0.99))").alias("abs_p50_p90_p99"),
        ).show(truncate=False)

# a few example rows
x.filter("d_cargo != 0").select(
    "portid", "date", "export_container", "export_dry_bulk", "export_general_cargo",
    "export_roro", "export_cargo", "d_cargo"
).show(5, truncate=False)

Py4JJavaError: An error occurred while calling o201.parquet.
: java.lang.NoClassDefFoundError: com/amazonaws/auth/AWSCredentialsProvider
	at java.base/java.lang.Class.forName0(Native Method)
	at java.base/java.lang.Class.forName(Class.java:467)
	at org.apache.hadoop.conf.Configuration.getClassByNameOrNull(Configuration.java:2625)
	at org.apache.hadoop.conf.Configuration.getClassByName(Configuration.java:2590)
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2686)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3431)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.exe

In [22]:
spark.stop()